In [32]:
import boto3

bucket = "dcceew-eds-data"
base_prefix = "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/"

s3 = boto3.client("s3")

# 1. list tile folders only
tile_resp = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=base_prefix,
    Delimiter="/"
)

tiles = [p["Prefix"] for p in tile_resp.get("CommonPrefixes", [])]

print("Tiles found:")
for t in tiles:
    print(t)

# 2. for each tile, check only immediate subfolders for output/outputs
print("\nTiles with output folders:\n")

for tile_prefix in tiles:
    sub_resp = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=tile_prefix,
        Delimiter="/"
    )
    subfolders = [p["Prefix"] for p in sub_resp.get("CommonPrefixes", [])]

    for sub in subfolders:
        name = sub.rstrip("/").split("/")[-1].lower()
        if name in {"output", "outputs"}:
            print(sub)

Tiles found:
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r081/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r082/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r084/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r078/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r080/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r081/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r082/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r083/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r084/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r085/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r086/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor

In [33]:
import boto3
import pandas as pd

bucket = "dcceew-eds-data"

prefixes = [
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r081/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r082/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/",
]

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

rows = []

for prefix in prefixes:
    shp_files = []
    tif_files = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            name = key.split("/")[-1]

            if key.lower().endswith(".shp"):
                shp_files.append(name)
            elif key.lower().endswith((".tif", ".tiff")):
                tif_files.append(name)

    tile = prefix.rstrip("/").split("/")[-2]

    rows.append({
        "tile": tile,
        "prefix": prefix,
        "n_shp": len(shp_files),
        "n_tif": len(tif_files),
        "shp_files": ", ".join(sorted(shp_files)) if shp_files else "",
        "tif_files": ", ".join(sorted(tif_files)) if tif_files else "",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

    tile                                                          prefix  n_shp  n_tif                                                                                                                           shp_files                                                                                                                                                                                                                                 tif_files
p089r080 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/      2      4 sl8olre_p089r080_d2025060720260109_dlj-dlj-clear-ge80_e32756.shp, sl8olre_p089r080_d2025060720260109_dlj-dlj-strong-ge60_e32756.shp sl8olre_p089r080_d2025060720260109_dlj-dlj-clear-ge80_e32756.tif, sl8olre_p089r080_d2025060720260109_dlj-dlj-strong-ge60_e32756.tif, sl8olre_p089r080_d2025060720260109_dlj_e32756.tif, sl8olre_p089r080_d2025060720260109_dll_e32756.tif
p089r081 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r081/outputs/      2      4 sl8olre

In [34]:
import boto3
import pandas as pd

bucket = "dcceew-eds-data"

prefixes = [
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r081/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r082/outputs/",
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/",
]

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

rows = []

for prefix in prefixes:
    n_shp = 0
    n_tif = 0

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"].lower()
            if key.endswith(".shp"):
                n_shp += 1
            elif key.endswith((".tif", ".tiff")):
                n_tif += 1

    tile = prefix.rstrip("/").split("/")[-2]
    rows.append({"tile": tile, "n_shp": n_shp, "n_tif": n_tif})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

    tile  n_shp  n_tif
p089r080      2      4
p089r081      2      4
p089r082      2      4
p089r083      2      4


In [35]:
import boto3
import pandas as pd

bucket = "dcceew-eds-data"

prefixes = [
    "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/",
]

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

rows = []

for prefix in prefixes:
    tile = prefix.rstrip("/").split("/")[-2]

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            low = key.lower()

            if low.endswith(".shp"):
                ext = ".shp"
            elif low.endswith(".tif") or low.endswith(".tiff"):
                ext = ".tif/.tiff"
            else:
                continue

            rows.append({
                "tile": tile,
                "ext": ext,
                "file": key.split("/")[-1],
                "size_mb": round(obj["Size"] / 1024 / 1024, 3),
                "full_key": key,
            })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

    tile        ext                                                              file  size_mb                                                                                                                                                                        full_key
p089r083 .tif/.tiff  sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.tif    1.055                AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/p089r083_d2025091120260109/masks/sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.tif
p089r083 .tif/.tiff sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.tif    1.330               AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/p089r083_d2025091120260109/masks/sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.tif
p089r083 .tif/.tiff                 sl8olre_p089r083_d2025091120260109_dlj_e32756.tif   26.979                                     AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/o

In [36]:
import re
import pandas as pd

# df = dataframe from earlier listing step

def extract_product(name):
    """
    extracts product part:
    sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.tif
                                          ^^^^^^^^^^^^^^^^^
    """
    
    m = re.search(r'_d\d+_(.*?)_e\d+', name)
    if m:
        return m.group(1)
    else:
        return "unknown"


df["product"] = df["file"].apply(extract_product)

summary = (
    df
    .groupby(["tile", "product", "ext"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

summary.columns.name = None

# rename columns
summary = summary.rename(columns={
    ".shp": "n_shp",
    ".tif/.tiff": "n_tif"
})

# attach filenames
files_grouped = (
    df
    .groupby(["tile", "product", "ext"])["file"]
    .apply(lambda x: ", ".join(sorted(x)))
    .unstack()
    .reset_index()
)

result = summary.merge(files_grouped, on=["tile","product"])

print(result.to_string(index=False))

    tile             product  n_shp  n_tif                                                              .shp                                                        .tif/.tiff
p089r083                 dlj      0      1                                                               NaN                 sl8olre_p089r083_d2025091120260109_dlj_e32756.tif
p089r083  dlj-dlj-clear-ge80      1      1  sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.shp  sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.tif
p089r083 dlj-dlj-strong-ge60      1      1 sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.shp sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.tif
p089r083                 dll      0      1                                                               NaN                 sl8olre_p089r083_d2025091120260109_dll_e32756.tif


In [37]:
summary_cols = ["tile", "product", "n_shp", "n_tif"]
print(result[summary_cols].fillna("").to_string(index=False))

    tile             product  n_shp  n_tif
p089r083                 dlj      0      1
p089r083  dlj-dlj-clear-ge80      1      1
p089r083 dlj-dlj-strong-ge60      1      1
p089r083                 dll      0      1


In [38]:
clean = result.rename(columns={
    ".shp": "shp_file",
    ".tif/.tiff": "tif_file"
}).fillna("")

clean = clean[["tile", "product", "n_shp", "n_tif", "shp_file", "tif_file"]]

print(clean.to_string(index=False))

    tile             product  n_shp  n_tif                                                          shp_file                                                          tif_file
p089r083                 dlj      0      1                                                                                   sl8olre_p089r083_d2025091120260109_dlj_e32756.tif
p089r083  dlj-dlj-clear-ge80      1      1  sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.shp  sl8olre_p089r083_d2025091120260109_dlj-dlj-clear-ge80_e32756.tif
p089r083 dlj-dlj-strong-ge60      1      1 sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.shp sl8olre_p089r083_d2025091120260109_dlj-dlj-strong-ge60_e32756.tif
p089r083                 dll      0      1                                                                                   sl8olre_p089r083_d2025091120260109_dll_e32756.tif


In [39]:
!aws s3 ls s3://dcceew-rs-data/dcceew_202602_run/ --recursive | grep "_completed_checked"

2026-03-23 02:16:51      41396 dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.dbf
2026-03-23 02:16:51        397 dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.prj
2026-03-23 02:16:52     462600 dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp
2026-03-23 02:16:52       3124 dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shx
2026-03-17 02:18:14          9 dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.cpg
2026-03-17 02:18:14      11639 dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.dbf
2026-03-17 23:36:30        397 dcceew_202602_run/lzolr

In [40]:
!aws s3 ls s3://dcceew-rs-data/dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked/

In [41]:
!aws s3 ls s3://dcceew-rs-data/dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked/ --recursive | grep ".shp"

In [42]:
import boto3
import pandas as pd
import re

bucket = "dcceew-eds-data"
base_prefix = "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/"

s3 = boto3.client("s3")

# get tile folders
tile_resp = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=base_prefix,
    Delimiter="/"
)

tiles = [p["Prefix"] for p in tile_resp.get("CommonPrefixes", [])]

output_prefixes = []

for tile_prefix in tiles:

    sub_resp = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=tile_prefix,
        Delimiter="/"
    )

    for p in sub_resp.get("CommonPrefixes", []):

        name = p["Prefix"].rstrip("/").split("/")[-1].lower()

        if name in ("output", "outputs"):
            output_prefixes.append(p["Prefix"])

print("Found output folders:")
for p in output_prefixes:
    print(p)

Found output folders:
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r081/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r082/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r083/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r084/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r078/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r080/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r081/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r082/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r083/outputs/
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r084/outputs/
AROAZ6PFZYT4B4C7MN

In [43]:
rows = []

paginator = s3.get_paginator("list_objects_v2")

def extract_product(filename):
    """
    extract:
    dlj-dlj-clear-ge80
    dlj-dlj-strong-ge60
    dll
    dlj
    """
    
    m = re.search(r'_d\d+_(.*?)_e\d+', filename)

    if m:
        return m.group(1)

    return "unknown"


for prefix in output_prefixes:

    tile = prefix.split("/")[-2]

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):

        for obj in page.get("Contents", []):

            key = obj["Key"].lower()

            if not key.endswith((".shp",".tif",".tiff")):
                continue

            filename = key.split("/")[-1]

            rows.append({
                "tile": tile,
                "product": extract_product(filename),
                "ext": filename.split(".")[-1],
                "file": filename,
                "size_mb": round(obj["Size"]/1024/1024,2),
                "s3_path": obj["Key"]
            })


df = pd.DataFrame(rows)

In [44]:
df

,tile,product,ext,file,size_mb,s3_path
0,outputs,dlj-dlj-clear-ge80,tif,sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...,0.84,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1,outputs,dlj-dlj-strong-ge60,tif,sl8olre_p089r078_d2025070120251208_dlj-dlj-str...,0.91,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
2,outputs,dlj,tif,sl8olre_p089r078_d2025070120251208_dlj_e32756.tif,9.88,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
3,outputs,dll,tif,sl8olre_p089r078_d2025070120251208_dll_e32756.tif,2.58,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
4,outputs,dlj-dlj-clear-ge80,shp,sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...,3.92,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
...,...,...,...,...,...,...
259,outputs,dlj-dlj-strong-ge60,tif,sl9olre_p092r084_d2025111220260107_dlj-dlj-str...,1.97,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
260,outputs,dlj,tif,sl9olre_p092r084_d2025111220260107_dlj_e32755.tif,47.60,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
261,outputs,dll,tif,sl9olre_p092r084_d2025111220260107_dll_e32755.tif,10.13,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
262,outputs,dlj-dlj-clear-ge80,shp,sl9olre_p092r084_d2025111220260107_dlj-dlj-cle...,12.27,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...


In [45]:
import re
import boto3
import pandas as pd
import geopandas as gpd

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
eds_bucket = "dcceew-eds-data"
eds_base_prefix = "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/"

roshan_bucket = "dcceew-rs-data"
roshan_base_prefix = "dcceew_202602_run/"

products_to_test = {"dlj-dlj-clear-ge80", "dlj-dlj-strong-ge60"}

s3 = boto3.client("s3")


# -------------------------------------------------
# HELPERS
# -------------------------------------------------
def list_all_keys(bucket, prefix):
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            yield obj["Key"], obj["Size"]


def parse_eds_key(key):
    """
    Example:
    AROAZ6.../eds/tiles/p089r083/outputs/p089r083_d2025091120260109/vectors/clear_ge80/...
    """
    m = re.search(r"/tiles/(p\d{3}r\d{3})/outputs/\1_d(\d{8})(\d{8})/", key)
    if not m:
        return None

    tile = m.group(1)
    start_date = m.group(2)
    end_date = m.group(3)

    filename = key.split("/")[-1]
    pm = re.search(r"_d\d{16}_(.*?)_e\d+\.(shp|tif|tiff)$", filename, flags=re.IGNORECASE)
    if not pm:
        return None

    product = pm.group(1)
    ext = pm.group(2).lower()

    return {
        "tile": tile,
        "start_date": start_date,
        "end_date": end_date,
        "product": product,
        "ext": ext,
        "file": filename,
        "key": key,
    }


def parse_roshan_key(key):
    """
    Example:
    dcceew_202602_run/lzolre_p106r069_d2025081820251013_dlwm2_completed/lzolre_p106r069_d2025081820251013_dlwm2_completed_checked.shp
    """
    filename = key.split("/")[-1]
    m = re.search(
        r"(p\d{3}r\d{3})_d(\d{8})(\d{8})_dlwm\d+_completed_checked\.shp$",
        filename,
        flags=re.IGNORECASE
    )
    if not m:
        return None

    return {
        "tile": m.group(1).lower(),
        "start_date": m.group(2),
        "end_date": m.group(3),
        "file": filename,
        "key": key,
    }


def s3_vsi_path(bucket, key):
    return f"/vsis3/{bucket}/{key}"


def normalise_isclearing(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
    )


# -------------------------------------------------
# 1. INDEX MY EDS OUTPUTS
# -------------------------------------------------
eds_rows = []

for key, size in list_all_keys(eds_bucket, eds_base_prefix):
    if not key.lower().endswith((".shp", ".tif", ".tiff")):
        continue

    parsed = parse_eds_key(key)
    if not parsed:
        continue

    if parsed["product"] not in products_to_test:
        continue

    parsed["size_mb"] = round(size / 1024 / 1024, 3)
    eds_rows.append(parsed)

eds_df = pd.DataFrame(eds_rows)

print("EDS candidate files:")
print(eds_df.head())
print(f"\nTotal EDS candidate rows: {len(eds_df)}")


# -------------------------------------------------
# 2. INDEX ROSHAN CHECKED SHAPEFILES
# -------------------------------------------------
roshan_rows = []

for key, size in list_all_keys(roshan_bucket, roshan_base_prefix):
    if not key.lower().endswith("_completed_checked.shp"):
        continue

    parsed = parse_roshan_key(key)
    if not parsed:
        continue

    parsed["size_mb"] = round(size / 1024 / 1024, 3)
    roshan_rows.append(parsed)

roshan_df = pd.DataFrame(roshan_rows)

print("\nRoshan checked shapefiles:")
print(roshan_df.head())
print(f"\nTotal Roshan checked shapefiles: {len(roshan_df)}")


# -------------------------------------------------
# 3. MATCH BY TILE + EXACT DATE WINDOW
# -------------------------------------------------
matched = eds_df.merge(
    roshan_df,
    on=["tile", "start_date", "end_date"],
    how="inner",
    suffixes=("_eds", "_roshan")
)

print("\nMatched tile/date rows:")
print(matched[[
    "tile", "start_date", "end_date", "product",
    "file_eds", "file_roshan"
]].to_string(index=False))

if matched.empty:
    raise ValueError("No exact tile/date matches found between EDS outputs and Roshan checked shapefiles.")


# -------------------------------------------------
# 4. COMPARE EACH MATCH
# -------------------------------------------------
results = []

for _, row in matched.iterrows():
    tile = row["tile"]
    start_date = row["start_date"]
    end_date = row["end_date"]
    product = row["product"]

    eds_path = s3_vsi_path(eds_bucket, row["key_eds"])
    roshan_path = s3_vsi_path(roshan_bucket, row["key_roshan"])

    try:
        eds_gdf = gpd.read_file(eds_path)
        roshan_gdf = gpd.read_file(roshan_path)
    except Exception as e:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": f"read_error: {e}",
        })
        continue

    # find IsClearing field
    isclearing_field = None
    for c in roshan_gdf.columns:
        if c.lower() == "isclearing":
            isclearing_field = c
            break

    if isclearing_field is None:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": "missing_IsClearing_field",
        })
        continue

    roshan_clear = roshan_gdf[normalise_isclearing(roshan_gdf[isclearing_field]) == "y"].copy()

    if roshan_clear.empty:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": "no_roshan_clearing_y",
        })
        continue

    # reproject to common projected CRS
    if eds_gdf.crs is None and roshan_clear.crs is None:
        target_crs = "EPSG:3577"
        eds_gdf = eds_gdf.set_crs(target_crs, allow_override=True)
        roshan_clear = roshan_clear.set_crs(target_crs, allow_override=True)
    elif eds_gdf.crs is None:
        eds_gdf = eds_gdf.set_crs(roshan_clear.crs, allow_override=True)
    elif roshan_clear.crs is None:
        roshan_clear = roshan_clear.set_crs(eds_gdf.crs, allow_override=True)

    # use EDS CRS if projected, else use 3577
    eds_crs_str = str(eds_gdf.crs).lower() if eds_gdf.crs else ""
    if "4326" in eds_crs_str:
        target_crs = "EPSG:3577"
    else:
        target_crs = eds_gdf.crs

    eds_gdf = eds_gdf.to_crs(target_crs)
    roshan_clear = roshan_clear.to_crs(target_crs)

    # remove empty geometries
    eds_gdf = eds_gdf[eds_gdf.geometry.notnull() & ~eds_gdf.geometry.is_empty].copy()
    roshan_clear = roshan_clear[roshan_clear.geometry.notnull() & ~roshan_clear.geometry.is_empty].copy()

    if eds_gdf.empty:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": "empty_eds_geometry",
        })
        continue

    # polygon-level strike rate
    roshan_clear = roshan_clear.reset_index(drop=True).copy()
    roshan_clear["roshan_id"] = roshan_clear.index
    roshan_clear["roshan_area_m2"] = roshan_clear.geometry.area

    eds_gdf = eds_gdf.reset_index(drop=True).copy()
    eds_gdf["eds_id"] = eds_gdf.index
    eds_gdf["eds_area_m2"] = eds_gdf.geometry.area

    try:
        inter = gpd.overlay(
            roshan_clear[["roshan_id", "roshan_area_m2", "geometry"]],
            eds_gdf[["eds_id", "eds_area_m2", "geometry"]],
            how="intersection"
        )
    except Exception as e:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": f"overlay_error: {e}",
        })
        continue

    if inter.empty:
        results.append({
            "tile": tile,
            "start_date": start_date,
            "end_date": end_date,
            "product": product,
            "status": "ok",
            "roshan_polygons": len(roshan_clear),
            "eds_polygons": len(eds_gdf),
            "roshan_hit_polygons": 0,
            "roshan_hit_rate": 0.0,
            "roshan_area_m2": roshan_clear["roshan_area_m2"].sum(),
            "intersect_area_m2": 0.0,
            "roshan_area_coverage": 0.0,
            "eds_area_m2": eds_gdf["eds_area_m2"].sum(),
            "eds_precision_like": 0.0,
        })
        continue

    inter["intersect_area_m2"] = inter.geometry.area

    # Roshan hit rate
    roshan_hits = (
        inter.groupby("roshan_id", as_index=False)["intersect_area_m2"]
        .sum()
        .merge(
            roshan_clear[["roshan_id", "roshan_area_m2"]],
            on="roshan_id",
            how="right"
        )
    )
    roshan_hits["intersect_area_m2"] = roshan_hits["intersect_area_m2"].fillna(0)
    roshan_hits["overlap_prop"] = roshan_hits["intersect_area_m2"] / roshan_hits["roshan_area_m2"]
    roshan_hit_polygons = (roshan_hits["intersect_area_m2"] > 0).sum()
    roshan_hit_rate = roshan_hit_polygons / len(roshan_hits)

    # area-based metrics
    roshan_area_m2 = roshan_clear["roshan_area_m2"].sum()
    eds_area_m2 = eds_gdf["eds_area_m2"].sum()
    intersect_area_m2 = inter["intersect_area_m2"].sum()

    roshan_area_coverage = intersect_area_m2 / roshan_area_m2 if roshan_area_m2 > 0 else None
    eds_precision_like = intersect_area_m2 / eds_area_m2 if eds_area_m2 > 0 else None

    results.append({
        "tile": tile,
        "start_date": start_date,
        "end_date": end_date,
        "product": product,
        "status": "ok",
        "roshan_polygons": len(roshan_clear),
        "eds_polygons": len(eds_gdf),
        "roshan_hit_polygons": int(roshan_hit_polygons),
        "roshan_hit_rate": float(roshan_hit_rate),
        "roshan_area_m2": float(roshan_area_m2),
        "intersect_area_m2": float(intersect_area_m2),
        "roshan_area_coverage": float(roshan_area_coverage) if roshan_area_coverage is not None else None,
        "eds_area_m2": float(eds_area_m2),
        "eds_precision_like": float(eds_precision_like) if eds_precision_like is not None else None,
        "eds_key": row["key_eds"],
        "roshan_key": row["key_roshan"],
    })

results_df = pd.DataFrame(results).sort_values(["tile", "start_date", "product"])

print("\nStrike-rate results:")
print(results_df.to_string(index=False))

EDS candidate files:
       tile start_date  end_date              product  ext  \
0  p089r078   20250701  20260117   dlj-dlj-clear-ge80  tif   
1  p089r078   20250701  20260117  dlj-dlj-strong-ge60  tif   
2  p089r078   20250701  20260117   dlj-dlj-clear-ge80  shp   
3  p089r078   20250701  20260117  dlj-dlj-strong-ge60  shp   
4  p089r079   20250607  20260125   dlj-dlj-clear-ge80  tif   

                                                file  \
0  sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...   
1  sl8olre_p089r078_d2025070120251208_dlj-dlj-str...   
2  sl8olre_p089r078_d2025070120251208_dlj-dlj-cle...   
3  sl8olre_p089r078_d2025070120251208_dlj-dlj-str...   
4  sl8olre_p089r079_d2025060720260109_dlj-dlj-cle...   

                                                 key  size_mb  
0  AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...    0.842  
1  AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...    0.909  
2  AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...    3.916  
3  AROAZ6PFZY

In [46]:
tile_test = "p090r077"
start_test = "20250817"
end_test = "20260201"

test_rows = matched[
    (matched.tile == tile_test) &
    (matched.start_date == start_test) &
    (matched.end_date == end_test) &
    (matched.file_eds.str.lower().str.endswith(".shp"))
].copy()

print(test_rows[[
    "tile",
    "start_date",
    "end_date",
    "product",
    "file_eds",
    "file_roshan"
]].to_string(index=False))

    tile start_date end_date             product                                                          file_eds                                                   file_roshan
p090r077   20250817 20260201  dlj-dlj-clear-ge80  sl9olre_p090r077_d2025081720260108_dlj-dlj-clear-ge80_e32756.shp lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp
p090r077   20250817 20260201 dlj-dlj-strong-ge60 sl9olre_p090r077_d2025081720260108_dlj-dlj-strong-ge60_e32756.shp lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp


In [47]:
import geopandas as gpd

roshan_key = test_rows.iloc[0]["key_roshan"]

roshan_path = f"/vsis3/{roshan_bucket}/{roshan_key}"

roshan = gpd.read_file(roshan_path)

# locate IsClearing column safely
isclearing_col = [c for c in roshan.columns if c.lower() == "isclearing"][0]

roshan_clear = roshan[
    roshan[isclearing_col].astype(str).str.lower() == "y"
].copy()

print("Roshan polygons:", len(roshan))
print("Roshan IsClearing='y':", len(roshan_clear))
print("CRS:", roshan.crs)

Roshan polygons: 378
Roshan IsClearing='y': 14
CRS: EPSG:32756


In [48]:
results = []

for _, r in test_rows.iterrows():

    eds_path = f"/vsis3/{eds_bucket}/{r.key_eds}"

    eds = gpd.read_file(eds_path)

    # ensure same CRS
    eds = eds.to_crs(roshan_clear.crs)

    # remove empty geometries
    eds = eds[eds.geometry.notnull()]
    roshan_clear = roshan_clear[roshan_clear.geometry.notnull()]

    # intersection
    inter = gpd.overlay(
        roshan_clear,
        eds,
        how="intersection"
    )

    roshan_area = roshan_clear.area.sum()
    eds_area = eds.area.sum()
    intersect_area = inter.area.sum()

    results.append({
        "tile": r.tile,
        "product": r.product,

        "roshan_polygons": len(roshan_clear),
        "eds_polygons": len(eds),

        "intersection_polygons": len(inter),

        "roshan_area_hit_%": intersect_area / roshan_area * 100,
        "eds_area_precision_%": intersect_area / eds_area * 100
    })

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

    tile                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [49]:
clean_rows = []

for _, r in test_rows.iterrows():

    eds_path = f"/vsis3/{eds_bucket}/{r.key_eds}"
    roshan_path = f"/vsis3/{roshan_bucket}/{r.key_roshan}"

    eds = gpd.read_file(eds_path)
    roshan = gpd.read_file(roshan_path)

    isclearing_col = [c for c in roshan.columns if c.lower()=="isclearing"][0]

    roshan_clear = roshan[
        roshan[isclearing_col].astype(str).str.lower()=="y"
    ].copy()

    eds = eds.to_crs(roshan_clear.crs)

    inter = gpd.overlay(roshan_clear, eds, how="intersection")

    roshan_area = roshan_clear.area.sum()
    eds_area = eds.area.sum()
    intersect_area = inter.area.sum()

    clean_rows.append({
        "tile": r["tile"],
        "start": r["start_date"],
        "end": r["end_date"],
        "product": r["product"],

        "roshan_polygons": len(roshan_clear),
        "eds_polygons": len(eds),

        "intersection_polygons": len(inter),

        "roshan_area_hit_%": round(intersect_area / roshan_area * 100, 2),
        "eds_precision_%": round(intersect_area / eds_area * 100, 2)
    })


results_df = pd.DataFrame(clean_rows)

results_df = results_df.sort_values(["tile","product"])

results_df

,tile,start,end,product,roshan_polygons,eds_polygons,intersection_polygons,roshan_area_hit_%,eds_precision_%
0,p090r077,20250817,20260201,dlj-dlj-clear-ge80,14,365,2,14.20,0.20
1,p090r077,20250817,20260201,dlj-dlj-strong-ge60,14,528,2,14.57,0.13


In [50]:
import geopandas as gpd
import pandas as pd

# only shapefiles for the 2 products
matched_all = matched[
    matched["file_eds"].str.lower().str.endswith(".shp") &
    matched["product"].isin([
        "dlj-dlj-clear-ge80",
        "dlj-dlj-strong-ge60"
    ])
].copy()

print(f"{len(matched_all)} shapefile matches found")

results = []

for i, r in matched_all.iterrows():

    print(
        f"processing {r.tile} {r.start_date}-{r.end_date} {r.product}"
    )

    eds_path = f"/vsis3/{eds_bucket}/{r.key_eds}"
    roshan_path = f"/vsis3/{roshan_bucket}/{r.key_roshan}"

    try:

        eds = gpd.read_file(eds_path)
        roshan = gpd.read_file(roshan_path)

        isclearing_col = [
            c for c in roshan.columns
            if c.lower()=="isclearing"
        ][0]

        roshan_clear = roshan[
            roshan[isclearing_col]
            .astype(str)
            .str.lower()
            == "y"
        ].copy()

        if len(roshan_clear)==0:

            results.append({
                "tile": r.tile,
                "start": r.start_date,
                "end": r.end_date,
                "product": r.product,
                "status": "no_roshan_y"
            })

            continue

        # ensure same CRS
        eds = eds.to_crs(roshan_clear.crs)

        # remove empty geometry
        eds = eds[eds.geometry.notnull()]
        roshan_clear = roshan_clear[
            roshan_clear.geometry.notnull()
        ]

        inter = gpd.overlay(
            roshan_clear,
            eds,
            how="intersection"
        )

        roshan_area = roshan_clear.area.sum()
        eds_area = eds.area.sum()
        intersect_area = inter.area.sum()

        results.append({

            "tile": r.tile,
            "start": r.start_date,
            "end": r.end_date,

            "product": r.product,

            "roshan_polygons": len(roshan_clear),
            "eds_polygons": len(eds),
            "intersection_polygons": len(inter),

            "roshan_area_hit_%":
                intersect_area / roshan_area * 100
                if roshan_area>0 else 0,

            "eds_precision_%":
                intersect_area / eds_area * 100
                if eds_area>0 else 0,

            "status": "ok"
        })

    except Exception as e:

        results.append({

            "tile": r.tile,
            "start": r.start_date,
            "end": r.end_date,
            "product": r.product,

            "status": str(e)
        })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["tile","start","product"]
)

results_df

8 shapefile matches found
processing p090r077 20250817-20260201 <bound method Series.prod of tile                                                       p090r077
start_date                                                 20250817
end_date                                                   20260201
product                                          dlj-dlj-clear-ge80
ext                                                             shp
file_eds          sl9olre_p090r077_d2025081720260108_dlj-dlj-cle...
key_eds           AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
size_mb_eds                                                    0.69
file_roshan       lzolre_p090r077_d2025081720260201_dlwm6_comple...
key_roshan        dcceew_202602_run/lzolre_p090r077_d20250817202...
size_mb_roshan                                                0.441
Name: 2, dtype: object>
processing p090r077 20250817-20260201 <bound method Series.prod of tile                                                       p090r077


TypeError: 'values' is not ordered, please explicitly specify the categories order by passing in a categories argument.

In [ ]:
summary = results_df[
    [
        "tile",
        "start",
        "end",
        "product",
        "roshan_polygons",
        "eds_polygons",
        "intersection_polygons",
        "roshan_area_hit_%",
        "eds_precision_%"
    ]
]

print(summary.to_string(index=False))

In [ ]:
overall = (
    summary
    .groupby("product")
    [
        [
            "roshan_area_hit_%",
            "eds_precision_%"
        ]
    ]
    .mean()
    .reset_index()
)

overall

In [ ]:
import geopandas as gpd
import pandas as pd

matched_all = matched[
    matched["file_eds"].str.lower().str.endswith(".shp") &
    matched["product"].isin([
        "dlj-dlj-clear-ge80",
        "dlj-dlj-strong-ge60"
    ])
].copy()

print(f"{len(matched_all)} shapefile matches found")

results = []

for _, r in matched_all.iterrows():
    tile = r["tile"]
    start_date = r["start_date"]
    end_date = r["end_date"]
    product = r["product"]

    print(f"processing {tile} {start_date}-{end_date} {product}")

    eds_path = f"/vsis3/{eds_bucket}/{r['key_eds']}"
    roshan_path = f"/vsis3/{roshan_bucket}/{r['key_roshan']}"

    try:
        eds = gpd.read_file(eds_path)
        roshan = gpd.read_file(roshan_path)

        isclearing_cols = [c for c in roshan.columns if c.lower() == "isclearing"]
        if not isclearing_cols:
            results.append({
                "tile": tile,
                "start": start_date,
                "end": end_date,
                "product": product,
                "status": "missing_IsClearing"
            })
            continue

        isclearing_col = isclearing_cols[0]

        roshan_clear = roshan[
            roshan[isclearing_col].astype(str).str.strip().str.lower() == "y"
        ].copy()

        if len(roshan_clear) == 0:
            results.append({
                "tile": tile,
                "start": start_date,
                "end": end_date,
                "product": product,
                "status": "no_roshan_y"
            })
            continue

        if eds.crs is None and roshan_clear.crs is not None:
            eds = eds.set_crs(roshan_clear.crs, allow_override=True)
        elif roshan_clear.crs is None and eds.crs is not None:
            roshan_clear = roshan_clear.set_crs(eds.crs, allow_override=True)

        if eds.crs != roshan_clear.crs:
            eds = eds.to_crs(roshan_clear.crs)

        eds = eds[eds.geometry.notnull() & ~eds.geometry.is_empty].copy()
        roshan_clear = roshan_clear[
            roshan_clear.geometry.notnull() & ~roshan_clear.geometry.is_empty
        ].copy()

        if len(eds) == 0:
            results.append({
                "tile": tile,
                "start": start_date,
                "end": end_date,
                "product": product,
                "status": "empty_eds"
            })
            continue

        inter = gpd.overlay(roshan_clear, eds, how="intersection")

        roshan_area = roshan_clear.geometry.area.sum()
        eds_area = eds.geometry.area.sum()
        intersect_area = inter.geometry.area.sum()

        results.append({
            "tile": tile,
            "start": start_date,
            "end": end_date,
            "product": product,
            "roshan_polygons": int(len(roshan_clear)),
            "eds_polygons": int(len(eds)),
            "intersection_polygons": int(len(inter)),
            "roshan_area_hit_pct": round((intersect_area / roshan_area * 100), 2) if roshan_area > 0 else 0.0,
            "eds_precision_pct": round((intersect_area / eds_area * 100), 2) if eds_area > 0 else 0.0,
            "status": "ok"
        })

    except Exception as e:
        results.append({
            "tile": tile,
            "start": start_date,
            "end": end_date,
            "product": product,
            "status": f"error: {e}"
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["tile", "start", "product"],
    ascending=[True, True, True]
).reset_index(drop=True)

results_df

In [ ]:
print(
    results_df[
        [
            "tile",
            "start",
            "end",
            "product",
            "roshan_polygons",
            "eds_polygons",
            "intersection_polygons",
            "roshan_area_hit_pct",
            "eds_precision_pct",
            "status",
        ]
    ].to_string(index=False)
)

In [ ]:
overall = (
    results_df[results_df["status"] == "ok"]
    .groupby("product")[["roshan_area_hit_pct", "eds_precision_pct"]]
    .mean()
    .reset_index()
)

print(overall.to_string(index=False))

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------
tile_test = "p089r084"
start_test = "20250911"
end_test = "20260109"

eds_bucket = "dcceew-eds-data"
roshan_bucket = "dcceew-rs-data"

# --------------------------------------------------
# 1. GET THE EXACT ROSHAN SHAPEFILE KEY FROM matched
# --------------------------------------------------
test_rows = matched[
    (matched["tile"] == tile_test) &
    (matched["start_date"] == start_test) &
    (matched["end_date"] == end_test)
].copy()

print("Matched rows:")
print(
    test_rows[
        ["tile", "start_date", "end_date", "product", "file_eds", "key_eds", "file_roshan", "key_roshan"]
    ].to_string(index=False)
)

if test_rows.empty:
    raise ValueError("No exact match found in matched for this tile/date window.")

roshan_key = test_rows.iloc[0]["key_roshan"]
roshan_path = f"/vsis3/{roshan_bucket}/{roshan_key}"

print("\nRoshan path:")
print(roshan_path)

# --------------------------------------------------
# 2. BUILD THE EXACT DLL TIFF KEY
# --------------------------------------------------
dll_filename = f"sl8olre_{tile_test}_d{start_test}{end_test}_dll_e32756.tif"
dll_key = f"AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/{tile_test}/outputs/{tile_test}_d{start_test}{end_test}/{dll_filename}"
dll_path = f"/vsis3/{eds_bucket}/{dll_key}"

print("\nDLL path:")
print(dll_path)

# --------------------------------------------------
# 3. LOAD ROSHAN AND FILTER IsClearing == 'y'
# --------------------------------------------------
roshan = gpd.read_file(roshan_path)

isclearing_cols = [c for c in roshan.columns if c.lower() == "isclearing"]
if not isclearing_cols:
    raise ValueError(f"No IsClearing field found. Columns are: {list(roshan.columns)}")

isclearing_col = isclearing_cols[0]

roshan_y = roshan[
    roshan[isclearing_col].astype(str).str.strip().str.lower() == "y"
].copy()

print("\nRoshan columns:")
print(list(roshan.columns))

print("\nIsClearing counts:")
print(
    roshan[isclearing_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts(dropna=False)
)

print(f"\nRoshan IsClearing='y' polygons: {len(roshan_y)}")

if roshan_y.empty:
    raise ValueError("No Roshan polygons where IsClearing == 'y'.")

# --------------------------------------------------
# 4. TEST DLL CLASSES INSIDE ROSHAN CLEARING POLYGONS
# DLL meanings:
#   0 = null/nodata
#   3 = non-clearing class
#   10 = no change
#   34-39 = clearing threshold classes
# --------------------------------------------------
rows = []

with rasterio.open(dll_path) as src:
    roshan_y = roshan_y.to_crs(src.crs)

    pixel_area_m2 = abs(src.transform.a * src.transform.e)

    for idx, feat in roshan_y.reset_index(drop=True).iterrows():
        geom = [feat.geometry.__geo_interface__]

        arr, _ = mask(src, geom, crop=True, filled=True, nodata=0)
        a = arr[0]

        vals, counts = np.unique(a, return_counts=True)
        counts_dict = dict(zip(vals.tolist(), counts.tolist()))

        total_px = int((a != 0).sum())

        row = {
            "roshan_id": idx,
            "valid_px": total_px,
            "count_0": counts_dict.get(0, 0),
            "count_3": counts_dict.get(3, 0),
            "count_10": counts_dict.get(10, 0),
            "count_34": counts_dict.get(34, 0),
            "count_35": counts_dict.get(35, 0),
            "count_36": counts_dict.get(36, 0),
            "count_37": counts_dict.get(37, 0),
            "count_38": counts_dict.get(38, 0),
            "count_39": counts_dict.get(39, 0),
        }

        threshold_px = sum(row[f"count_{c}"] for c in [34, 35, 36, 37, 38, 39])

        row["threshold_px_total"] = threshold_px
        row["threshold_hit"] = threshold_px > 0
        row["threshold_hit_pct_of_polygon"] = round((threshold_px / total_px * 100), 2) if total_px > 0 else 0.0

        rows.append(row)

hits_df = pd.DataFrame(rows)

print("\nPer-polygon hits:")
print(hits_df.to_string(index=False))

# --------------------------------------------------
# 5. SUMMARY
# --------------------------------------------------
summary = pd.DataFrame([{
    "tile": tile_test,
    "start": start_test,
    "end": end_test,
    "roshan_polygons_y": len(hits_df),
    "polygons_with_any_threshold_hit": int(hits_df["threshold_hit"].sum()),
    "strike_rate_pct": round(hits_df["threshold_hit"].mean() * 100, 2) if len(hits_df) else 0.0,
    "sum_count_3": int(hits_df["count_3"].sum()),
    "sum_count_10": int(hits_df["count_10"].sum()),
    "sum_count_34": int(hits_df["count_34"].sum()),
    "sum_count_35": int(hits_df["count_35"].sum()),
    "sum_count_36": int(hits_df["count_36"].sum()),
    "sum_count_37": int(hits_df["count_37"].sum()),
    "sum_count_38": int(hits_df["count_38"].sum()),
    "sum_count_39": int(hits_df["count_39"].sum()),
    "sum_threshold_px_total": int(hits_df["threshold_px_total"].sum()),
}])

print("\nSummary:")
print(summary.to_string(index=False))

# --------------------------------------------------
# 6. CLASS SUMMARY
# --------------------------------------------------
class_summary = pd.DataFrame({
    "class": [34, 35, 36, 37, 38, 39],
    "pixels": [int(hits_df[f"count_{c}"].sum()) for c in [34, 35, 36, 37, 38, 39]],
    "hit_polygons": [int((hits_df[f"count_{c}"] > 0).sum()) for c in [34, 35, 36, 37, 38, 39]],
})

print("\nClass summary:")
print(class_summary.to_string(index=False))

In [ ]:
# What Roshan checked shapefiles exist for p089r084?
roshan_df[roshan_df["tile"] == "p089r084"].sort_values(["start_date", "end_date"])

In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path
from datetime import datetime


BUCKET = "dcceew-rs-data"
PREFIX = "dcceew_202602_run/"
HOME = Path.home()

# local folder where the S3 contents will be downloaded
LOCAL_DIR = HOME / "roshan_dcceew_202602_run"

# output zip file
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_PATH = HOME / f"roshan_dcceew_202602_run_{STAMP}.zip"


def run_cmd(cmd: list[str]) -> None:
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, check=False)
    if proc.returncode != 0:
        raise SystemExit(f"Command failed with return code {proc.returncode}")


def main() -> None:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)

    s3_uri = f"s3://{BUCKET}/{PREFIX}"

    print(f"Downloading from: {s3_uri}")
    print(f"Local folder:     {LOCAL_DIR}")
    print(f"Zip output:       {ZIP_PATH}")

    # download everything from S3
    run_cmd([
        "aws", "s3", "sync",
        s3_uri,
        str(LOCAL_DIR),
    ])

    # create zip (without .zip suffix passed to make_archive)
    zip_base = str(ZIP_PATH.with_suffix(""))

    print("Creating zip archive...")
    shutil.make_archive(
        base_name=zip_base,
        format="zip",
        root_dir=str(LOCAL_DIR.parent),
        base_dir=LOCAL_DIR.name,
    )

    print("\nDone.")
    print(f"Downloaded folder: {LOCAL_DIR}")
    print(f"Zip archive:       {ZIP_PATH}")


if __name__ == "__main__":
    main()

Local folder:     /home/jovyan/roshan_dcceew_202602_run
Zip output:       /home/jovyan/roshan_dcceew_202602_run_20260324_042459.zip
Running: aws s3 sync s3://dcceew-rs-data/dcceew_202602_run/ /home/jovyan/roshan_dcceew_202602_run
download: s3://dcceew-rs-data/dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.prj to ../../../roshan_dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.prj
download: s3://dcceew-rs-data/dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.shp to ../../../roshan_dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.shp
download: s3://dcceew-rs-data/dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.dbf to ../../../roshan_dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6.dbf
download: s